# Orthogonality

## What's covered

- **Orthogonal vectors** and **orthonormal sets** — the cleanest possible basis
- **Orthogonal complement** of a subspace
- **Projection onto a subspace** — the natural generalization of projecting onto a single vector
- **Gram-Schmidt** — turning any basis into an orthonormal one
- **QR decomposition** — Gram-Schmidt packaged as a matrix factorization
- **Least squares** — deriving the normal equations from projection
- Where this appears in ML — orthogonal weight init, PCA preprocessing, regression as projection


## Orthogonal vectors and orthonormal sets

Two vectors are **orthogonal** when their dot product is zero. Geometrically, the angle between them is 90 degrees. We have seen this several times — now we elevate it to a *design principle* for bases.

A set of vectors is **orthonormal** when:

1. Every vector has unit length (`||v_i|| = 1`).
2. Any two distinct vectors are orthogonal (`v_i · v_j = 0` for `i ≠ j`).

Why bother? Because an orthonormal basis makes nearly every computation cheap:

- **Coordinates are free.** If `{q_1, q_2, ..., q_n}` is orthonormal, then for any `x = c_1 q_1 + c_2 q_2 + ... + c_n q_n`, the coordinates are just `c_i = q_i · x`. No system to solve.
- **Projections are dot products.** Projecting `x` onto the line through `q_i` is `(q_i · x) q_i` — no `||q_i||^2` in the denominator because it equals 1.
- **An orthonormal matrix has `Q^T = Q^{-1}`.** The cheapest possible inverse.

In short: orthonormal bases are to vector spaces what `units` and `right angles` are to engineering drawings — they remove friction.


In [ ]:
import numpy as np

# Two orthonormal vectors
q1 = np.array([1.0, 0.0])
q2 = np.array([0.0, 1.0])
print("||q1|| =", np.linalg.norm(q1), "  ||q2|| =", np.linalg.norm(q2))
print("q1 . q2 =", q1 @ q2)

# Cheap coordinates trick: any x is just (q1.x)*q1 + (q2.x)*q2
x = np.array([3.0, 4.0])
coords = np.array([q1 @ x, q2 @ x])
print("coords =", coords, "  reconstruct:", coords[0]*q1 + coords[1]*q2)

# A rotated orthonormal basis at 30 degrees
theta = np.pi / 6
r1 = np.array([np.cos(theta), np.sin(theta)])
r2 = np.array([-np.sin(theta), np.cos(theta)])
print("\nrotated basis still orthonormal? ", np.allclose([r1 @ r1, r2 @ r2, r1 @ r2], [1, 1, 0]))


## Orthogonal complement

Given a subspace `S` of `R^n`, its **orthogonal complement** `S^⊥` (read "S perp") is the set of all vectors orthogonal to *every* vector in `S`:

$$
S^\perp = \{ \mathbf{v} \in \mathbb{R}^n \;|\; \mathbf{v} \cdot \mathbf{s} = 0 \text{ for every } \mathbf{s} \in S \}
$$

`S^⊥` is itself a subspace, and the two together fill all of `R^n` cleanly:

$$
\dim(S) + \dim(S^\perp) = n
$$

Every vector `x ∈ R^n` splits *uniquely* into a piece in `S` and a piece in `S^⊥`:

$$
\mathbf{x} = \mathbf{x}_S + \mathbf{x}_{S^\perp}
$$

We met this already in notebook 5 as the punchline of the four fundamental subspaces:

- `Row(A) ⊥ Null(A)` in `R^n`
- `Col(A) ⊥ Null(A^T)` in `R^m`

These are not coincidences — they are *the* defining picture of how a matrix splits its input and output spaces.


## Projection onto a subspace

In notebook 2 we projected a vector onto a single vector. We now generalize to projecting onto an entire subspace.

Given a subspace `S = Col(A)` (the column space of some `m × n` matrix `A`), the **projection of `b` onto `S`** is the unique vector `p ∈ S` that minimizes `||b - p||`. Geometrically: drop a perpendicular from `b` onto `S` and call the foot `p`. The leftover piece `b - p` lies entirely in `S^⊥`.

**The formula.** Writing `p = A\hat{x}` (so `p` is a linear combination of the columns of `A`, with weights `\hat{x}`), the orthogonality condition `A^T (b - A\hat{x}) = 0` rearranges to:

$$
A^T A \, \hat{\mathbf{x}} = A^T \mathbf{b}
$$

That's the **normal equations**, derived in one step. The projection is then `p = A\hat{x}`, and the explicit projection matrix is:

$$
P = A (A^T A)^{-1} A^T
$$

Two sanity properties: `P^2 = P` (projecting twice is the same as projecting once) and `P^T = P` (`P` is symmetric — true of every orthogonal projection).

**Special case: orthonormal columns.** If the columns of `A` are orthonormal, then `A^T A = I`, so the projection collapses to `P = A A^T` and the weights are `\hat{x} = A^T b`. This is *the* reason orthonormal bases are worth the trouble.


In [ ]:
# Project b onto the column space of A
A = np.array([[1.0, 0.0],
              [1.0, 1.0],
              [1.0, 2.0]])
b = np.array([1.0, 2.0, 2.0])

# Solve the normal equations (illustrative; lstsq is preferred numerically)
x_hat = np.linalg.solve(A.T @ A, A.T @ b)
p = A @ x_hat
print("projection p   =", p)
print("residual b - p =", b - p)
print("residual ⊥ Col(A)?", np.allclose(A.T @ (b - p), 0))

# The projection matrix
P = A @ np.linalg.inv(A.T @ A) @ A.T
print("\nP @ b =", P @ b, "  (matches p)")
print("P @ P = P?", np.allclose(P @ P, P))
print("P symmetric?", np.allclose(P, P.T))


## Gram-Schmidt — building an orthonormal basis

You rarely *start* with an orthonormal basis. Usually you start with whatever vectors you have and need to manufacture one. **Gram-Schmidt** is the standard procedure.

Given linearly independent vectors `a_1, a_2, ..., a_n`, produce orthonormal vectors `q_1, q_2, ..., q_n` with the same span:

1. Take `a_1`, normalize: `q_1 = a_1 / ||a_1||`.
2. Take `a_2`, subtract its projection onto `q_1` to make it orthogonal, then normalize:
   $$
   \mathbf{u}_2 = \mathbf{a}_2 - (\mathbf{q}_1 \cdot \mathbf{a}_2) \mathbf{q}_1, \qquad \mathbf{q}_2 = \mathbf{u}_2 / \|\mathbf{u}_2\|
   $$
3. In general, subtract the projections onto *all previous* `q`'s, then normalize:
   $$
   \mathbf{u}_k = \mathbf{a}_k - \sum_{j=1}^{k-1} (\mathbf{q}_j \cdot \mathbf{a}_k) \mathbf{q}_j, \qquad \mathbf{q}_k = \mathbf{u}_k / \|\mathbf{u}_k\|
   $$

That's it. Each step strips out the components already accounted for, leaving the new direction perpendicular to everything before it.

**Numerical caveat.** Classical Gram-Schmidt as written above accumulates floating-point error; for production use, prefer **modified Gram-Schmidt** or (much more commonly) call `np.linalg.qr`, which uses Householder reflections under the hood.


In [ ]:
# Classical Gram-Schmidt by hand on three vectors in R^3
a1 = np.array([1.0, 1.0, 0.0])
a2 = np.array([1.0, 0.0, 1.0])
a3 = np.array([0.0, 1.0, 1.0])

q1 = a1 / np.linalg.norm(a1)

u2 = a2 - (q1 @ a2) * q1
q2 = u2 / np.linalg.norm(u2)

u3 = a3 - (q1 @ a3) * q1 - (q2 @ a3) * q2
q3 = u3 / np.linalg.norm(u3)

Q = np.column_stack([q1, q2, q3])
print("Q =\n", Q)
print("\nQ^T Q =\n", Q.T @ Q, "  <- should be identity, confirming orthonormality")


## QR decomposition

Gram-Schmidt is so important that it gets a matrix-level name: **QR decomposition**. Any `m × n` matrix `A` with linearly independent columns factors as

$$
A = Q R
$$

where `Q` is `m × n` with orthonormal columns (`Q^T Q = I`) and `R` is `n × n` upper triangular. The columns of `Q` are exactly the `q_i` from Gram-Schmidt; `R` records how to combine them to get the original `a_i` back.

**Why this is useful.**

- **Stable least squares.** Substitute `A = QR` into the normal equations: `(QR)^T (QR) \hat{x} = (QR)^T b` ⟹ `R \hat{x} = Q^T b`. Solving an upper-triangular system is one back-substitution — fast and numerically excellent. This is what `np.linalg.lstsq` does internally.
- **Orthogonal weight initialization.** Initialize a weight matrix as `Q` from `QR` of a random matrix. Guarantees orthonormal columns at start, which keeps signal norms stable through deep networks.
- **Building blocks for eigenvalue algorithms.** The QR algorithm repeatedly factors `A = QR`, then sets `A ← RQ`. After many iterations `A` converges to an upper triangular matrix whose diagonal contains the eigenvalues. Beautiful and unreasonably effective.


In [ ]:
# QR decomposition via NumPy
A = np.array([[1.0, 1.0, 0.0],
              [1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0]])

Q, R = np.linalg.qr(A)
print("Q =\n", Q)
print("\nR =\n", R)
print("\nQ^T Q ≈ I?", np.allclose(Q.T @ Q, np.eye(3)))
print("Q @ R == A?", np.allclose(Q @ R, A))


## Least squares, via projection

Bringing it together. Suppose we want to fit `Ax ≈ b` where `b ∉ Col(A)` — the usual ML setup, more equations than unknowns. There is no exact solution, so we find the `\hat{x}` that minimizes the **squared error**:

$$
\hat{\mathbf{x}} = \arg\min_x \| A \mathbf{x} - \mathbf{b} \|_2^2
$$

The geometric trick: the closest reachable vector to `b` is its projection `p` onto `Col(A)`. So we want `A \hat{x} = p`. Since `b - A\hat{x}` is the *residual* — the unreachable part of `b` — it must be orthogonal to `Col(A)`:

$$
A^T (\mathbf{b} - A \hat{\mathbf{x}}) = 0
$$

Rearranging gives the **normal equations** one more time:

$$
A^T A \, \hat{\mathbf{x}} = A^T \mathbf{b}
$$

This is *the* equation of linear regression. Three equivalent ways to solve it:

1. **Normal equations directly** — `np.linalg.solve(A.T @ A, A.T @ b)`. Concise but squares the condition number. Risky for ill-conditioned `A`.
2. **QR** — `Q, R = qr(A); solve(R, Q.T @ b)`. Numerically stable. What `lstsq` does internally.
3. **SVD** — handles rank-deficient `A` correctly via the pseudoinverse. Slowest but most robust.

In practice: `np.linalg.lstsq` for one-off code; the QR or Cholesky paths inside scikit-learn / PyTorch internals.


In [ ]:
# Three ways to solve the same least-squares problem
rng = np.random.default_rng(42)
m, n = 20, 3
A = rng.normal(size=(m, n))
true_x = np.array([1.0, -2.0, 0.5])
b = A @ true_x + 0.1 * rng.normal(size=m)

# 1) Normal equations (illustrative)
x1 = np.linalg.solve(A.T @ A, A.T @ b)

# 2) QR
Q, R = np.linalg.qr(A)
x2 = np.linalg.solve(R, Q.T @ b)

# 3) lstsq (uses SVD internally)
x3, *_ = np.linalg.lstsq(A, b, rcond=None)

print("true x  :", true_x)
print("normal  :", x1)
print("QR      :", x2)
print("lstsq   :", x3)
print("\nall three agree?", np.allclose(x1, x2) and np.allclose(x2, x3))


## Where this appears in ML

Orthogonality is the single most useful structural property in numerical ML. Once a basis becomes orthonormal, half your problems vanish.

- **Linear regression.** Least squares = projection onto column space, solved via QR. Every linear regressor under the hood.
- **Ridge regression.** Adds `λ ||w||^2`, equivalent to projecting onto a slightly tilted subspace — keeps a unique solution even when `Col(A)` is rank-deficient.
- **PCA, whitening.** Express data in an orthonormal basis aligned with variance directions. Whitening then rescales so the covariance becomes `I`.
- **Orthogonal weight initialization.** Start weights as `Q` from `QR(random)`. Preserves signal norms through deep networks; critical for RNNs and very deep MLPs.
- **Self-attention math.** Queries, keys, and values are projections via learned matrices; output is a projection back into the embedding space.
- **Gram-Schmidt in modern systems.** Used to orthogonalize gradient updates (e.g., MUON optimizer), to construct orthonormal embeddings, and to maintain numerical conditioning during training.
- **Reflection / rotation layers in normalizing flows.** Built from orthogonal matrices because they have unit determinant — making the change-of-variables Jacobian trivial.

Next notebook: **eigendecomposition** — directions that a matrix only stretches (not rotates), and the diagonalization `A = P D P^{-1}` that makes them visible.
